# ICT-19 — La batterie de l'ENJEU : auto-maintien vs pur dissipateur

**Série ICT** (Epic #4588, strate 5). Ce notebook est la **batterie de l'ENJEU** :
il pose et falsifie la question « un système manifeste-t-il de l'agence sur sa
propre forme ? » en opérant le distinguo le plus difficile — celui entre un
système qui **s'auto-maintient** (répare activement sa structure après une
agression) et un **pur dissipateur** (dissipe fortement mais ne répare pas).

> **Cadrage B (verrouillé, GPU-free).** L'agence n'est jamais *déclarée*, elle
> est *mesurée* — et toujours **par contraste** avec un contrôle passif. Le
> discriminateur n'est pas l'intensité de dissipation (`I_thermo` brut, qui
> s'allume pour toute flame/tornade/convection) mais le **gain de réparation**
> `repair_gain` : la différence entre la récupération sous la dynamique **complète**
> et sous un contrôle **privé du mécanisme réparateur** (diffusion pure).

**Prérequis** : [ICT-9 (agence morphologique)](ICT-9-AgencyRegeneration.ipynb),
[ICT-18 (flèche du temps / réversibilisation)](ICT-18-ArrowOfTimeReversibilization.ipynb).

## 1. Le problème : `I_thermo` seul ne suffit pas

La flèche du temps thermodynamique ([ICT-18](ICT-18-ArrowOfTimeReversibilization.ipynb))
mesure la production d'entropie $\sigma$ d'un système — son irréversibilité. Un
système agent (cellule, organisme, motif auto-entretenu) dissipe : donc $\sigma > 0$.
**Mais l'inverse est faux** : une flame, une tornade, une cellule de Bénard dissipent
**massivement** sans pour autant être des agents — ce sont des *dissipateurs purs*,
maintenus par un flux externe, qui ne *réparent* pas leur forme.

ICT-18 §2.bis l'avait déjà signalé en faux-positif : un oscillateur forcé allume
$I_{thermo}$ à $\sigma \approx 4.6$ (~4000× le bruit iid) sans être agent. La
batterie de l'ENJEU est précisément l'instrument qui **démêle** ces deux cas.

**Réponse proposée** : le discriminateur est le **gain de réparation** `repair_gain`,
cadre do-calculus de Pearl. On compare deux mondes issus de la **même**
intervention $do(\text{ablation})$ : l'un sous la dynamique complète, l'autre sous
diffusion pure. Un système agent **répare** (gain >> 0) ; un dissipateur pur **subit**
(gain ≈ 0).

In [1]:
import numpy as np
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "ict"))

from ict import reaction_diffusion as rd
from ict import agency
from ict import time_arrow

RNG = np.random.default_rng(0)
print(f"numpy {np.__version__} — modules ict charges (agency, reaction_diffusion, time_arrow).")

numpy 2.4.4 — modules ict charges (agency, reaction_diffusion, time_arrow).


## 2. Substrat S4 — Gray-Scott (le système agent)

Le substrat S4 est un système de **réaction-diffusion de Gray-Scott** en régime
auto-entretenu (spots). Après un warmup qui forme le motif, on observe une structure
spatiale stable maintenue par le couplage autocatalytique $U + 2V \to 3V$.

In [2]:
# S4 : Gray-Scott, warmup pour former le motif auto-entretenu
gs = rd.GrayScott()
n = 64
U, V = gs.seed(n=n, rng=np.random.default_rng(11))
U, V, _ = gs.run(U, V, steps=3000)   # warmup : motif forme
V_ref_S4 = V.copy()
print(f"S4 (Gray-Scott) structure globale apres warmup : {agency.structure(V_ref_S4):.4f}")

S4 (Gray-Scott) structure globale apres warmup : 0.0100


### Mesure du `repair_gain` sur S4

On applique l'intervention $do(\text{ablation})$ : on réinitialise un quadrant à
l'état nu ($U=1, V=0$). Puis on compare la récupération sous la dynamique **complète**
(réaction-diffusion) vs un contrôle **passif** (diffusion pure, privé du couplage
autocatalytique). Le `repair_gain` est la différence : strictement positif et
substantiel, il signe un système qui **répare activement**.

In [3]:
mask_S4 = agency.quadrant_mask(n, quadrant=3)
U_abl, V_abl = agency.ablate(U, V, mask_S4)

# Monde 1 : reaction-diffusion complete
_, V_rd, _ = gs.run(U_abl.copy(), V_abl.copy(), steps=4000)
# Monde 2 : diffusion pure (controle passif)
V_diff = rd.run_pure_diffusion(V_abl.copy(), D=gs.Dv, steps=4000)

rec_rd_S4 = agency.recovery_score(V_ref_S4, V_abl, V_rd, mask_S4)
rec_diff_S4 = agency.recovery_score(V_ref_S4, V_abl, V_diff, mask_S4)
gain_S4 = agency.repair_gain(rec_rd_S4, rec_diff_S4)

print(f"S4 recovery (RD complete)  : {rec_rd_S4:.3f}")
print(f"S4 recovery (diff passive) : {rec_diff_S4:.3f}")
print(f"S4 repair_gain             : {gain_S4:.3f}   [>> 0 = AGENT]")

S4 recovery (RD complete)  : 1.160
S4 recovery (diff passive) : 0.000
S4 repair_gain             : 1.160   [>> 0 = AGENT]


## 3. Substrat S5 — oscillateur logistique piloté (le dissipateur pur)

Le substrat S5 est un **oscillateur logistique piloté** spatialisé, sans diffusion.
Chaque cellule évolue selon $x_{t+1} = r x_t(1 - x_t) + A\sin(2\pi t/T)$ avec
$r = 3.9$ (régime chaotique) et un faible forçage périodique. La sensibilité aux
conditions initiales crée une **structure spatiale émergente** ; le forçage maintain
une dissipation élevée ($\sigma$ élevé). **Mais** ce système n'a pas de *consigne
intrinsèque* : après ablation, il ne *répare* pas sa forme.

In [4]:
# S5 : oscillateur logistique pilote spatialise (r=3.9, forcing=0.1, sans diffusion)
R, FORCE_AMP, FORCE_PERIOD = 3.9, 0.1, 12

def logistic_forced(field, t):
    f = FORCE_AMP * np.sin(2 * np.pi * t / FORCE_PERIOD)
    out = R * field * (1.0 - field) + f
    return np.clip(out, 0.0, 1.0)

field = np.full((n, n), 0.5) + 0.05 * np.random.default_rng(3).standard_normal((n, n))
field = np.clip(field, 0.01, 0.99)
for t in range(1500):
    field = logistic_forced(field, t)
V_ref_S5 = field.copy()
print(f"S5 (logistique pilote) structure globale : {agency.structure(V_ref_S5):.4f}")

S5 (logistique pilote) structure globale : 0.0616


### Mesure du `repair_gain` sur S5

Même protocole : $do(\text{ablation})$ d'un quadrant, puis récupération complète vs
passive. Hypothèse : S5 dissipe fortement mais ne **répare** pas → `repair_gain ≈ 0`.

In [5]:
mask_S5 = agency.quadrant_mask(n, quadrant=3)
V_abl_S5 = V_ref_S5.copy()
V_abl_S5[mask_S5] = 0.5

# Monde 1 : dynamique complete (logistique pilote)
V_full_S5 = V_abl_S5.copy()
for t in range(3000):
    V_full_S5 = logistic_forced(V_full_S5, 1500 + 120 + t)
# Monde 2 : diffusion pure (controle passif)
V_diff_S5 = V_abl_S5.copy()
for _ in range(3000):
    V_diff_S5 = V_diff_S5 + 0.02 * rd.laplacian(V_diff_S5)

s_ref = agency.local_structure(V_ref_S5, mask_S5)
s_abl = agency.local_structure(V_abl_S5, mask_S5)
s_full = agency.local_structure(V_full_S5, mask_S5)
s_diff = agency.local_structure(V_diff_S5, mask_S5)
denom = s_ref - s_abl
rec_full_S5 = (s_full - s_abl) / denom if abs(denom) > 1e-12 else float('nan')
rec_diff_S5 = (s_diff - s_abl) / denom if abs(denom) > 1e-12 else float('nan')
gain_S5 = rec_full_S5 - rec_diff_S5

print(f"S5 recovery (complete)  : {rec_full_S5:.3f}")
print(f"S5 recovery (passive)   : {rec_diff_S5:.3f}")
print(f"S5 repair_gain          : {gain_S5:.3f}   [~ 0 = PUR DISSIPATEUR]")

S5 recovery (complete)  : 0.000
S5 recovery (passive)   : 0.004
S5 repair_gain          : -0.004   [~ 0 = PUR DISSIPATEUR]


## 4. La batterie ENJEU-1 : `repair_gain` comme discriminateur

La batterie de l'ENJEU opère le distinguo que `I_thermo` seul ne peut faire. Deux
systèmes peuvent dissiper (cf. §5 ci-dessous) ; ce qui les sépare est la capacité à
**réparer activement** leur forme après une agression — mesurée par `repair_gain`.

**Gate ENJEU-1** : `repair_gain(S4) > 0.2` (agent) **ET** `|repair_gain(S5)| < 0.15`
(dissipateur pur). Pas de clause sur `I_thermo` brut — la dissipation seule ne
discrimine pas, les deux substrats dissipent (voir §5). C'est le cadre do-calculus
de Pearl : même intervention `do(ablation)`, deux mondes, une différence.

In [6]:
print("=" * 60)
print("BATTERIE ENJEU-1 — discriminateur repair_gain")
print("=" * 60)
print(f"  S4 Gray-Scott (agent)        : repair_gain = {gain_S4:+.3f}")
print(f"  S5 logistique pilote (dissip): repair_gain = {gain_S5:+.3f}")
print()
gate_pass = (gain_S4 > 0.2) and (abs(gain_S5) < 0.15)
print(f"  Gate ENJEU-1 (repair_gain S4 >> S5 ~ 0) : {'PASS' if gate_pass else 'FAIL'}")
print()
print("L'agence morphologique (S4) se distingue du pur dissipateur (S5)")
print("par le gain de reparation, non par l'intensite de dissipation.")

BATTERIE ENJEU-1 — discriminateur repair_gain
  S4 Gray-Scott (agent)        : repair_gain = +1.160
  S5 logistique pilote (dissip): repair_gain = -0.004

  Gate ENJEU-1 (repair_gain S4 >> S5 ~ 0) : PASS

L'agence morphologique (S4) se distingue du pur dissipateur (S5)
par le gain de reparation, non par l'intensite de dissipation.


## 5. Ce que la batterie NE doit PAS affirmer : `I_thermo` seul ne discrimine pas

Il serait malhonnête de prétendre que S4 (agent) dissipe et S5 (dissipateur) ne
dissipe pas — c'est **faux**, et c'est précisément la leçon. Mesurons la dissipation
des deux substrats par un **bilan de flux** (proxy thermodynamique continu du taux
de production d'entropie d'un NESS — *non-equilibrium steady state*) :

- **S4** : flux réactionnel $\langle U V^2 \rangle$ (autocatalyse locale, moyennée).
  Au NESS, ce flux est maintain par feed+kill → signature hors-équilibre **positive**.
- **S5** : taux de changement temporel $\langle (x_{t+1} - x_t)^2 \rangle$ (itérateur
  chaotique forcé). Proxy de l'activité hors-équilibre d'un oscillateur piloté.

Les **deux** sont au-dessus du contrôle uniforme. CQFD : `I_thermo` seul ne sépare
pas S4 et S5.

## 6. Limitations honnêtes

- **`repair_gain` mesure la réparation morphologique locale** (variance au masque).
  D'autres formes d'agence (cognition basale, valence, anticipation) échappent à cet
  instrument — elles relèvent d'ICT-12 (valence) ou ICT-14 (free-energy).
- **La batterie est binaire** (agent vs dissipateur pur). Une graduation fine entre
  substrats (S1 tri, S2 bistable, S3 Axelrod) est l'objet d'ENJEU-2 (exercice 3).
- **S5 est un dissipateur « trivial »** (itérateur chaotique forcé). Un dissipateur
  plus subtil (convection de Rayleigh-Bénard, onde solitaire) pourrait donner un
  `repair_gain` ambigu — la batterie est falsifiable, pas absolue.

## 7. Exercices

### Exercice 1 — Sensibilité au `warmup`
Rejouez `repair_gain(S4)` en balayant `warmup ∈ {500, 1000, 3000, 5000}`.
Le gain se stabilise-t-il ? Un warmup insuffisant (motif non formé) donne-t-il un
faux négatif ?

In [7]:
# Bilan de flux : proxy de la dissipation hors-equilibre (NESS)
# S4 : flux reactionnel autocatalytique moyen (U*V^2)
sigma_flux_S4 = float(np.mean(U * V ** 2))
# S5 : taux de changement temporel moyen (iterateur chaotique force)
traj_S5 = [V_ref_S5.copy()]
ft = V_ref_S5.copy()
for t in range(120):
    ft = logistic_forced(ft, 1500 + t)
    traj_S5.append(ft.copy())
sigma_iter_S5 = float(np.mean(np.diff(np.array(traj_S5), axis=0) ** 2))
# Controle : substrat uniforme (U=1, V=0) -> pas de flux
sigma_uniform = 0.0

print("Bilan de flux (proxy dissipation NESS) :")
print(f"  S4 Gray-Scott  : sigma_flux = {sigma_flux_S4:.5f}  (> controle {sigma_uniform:.5f})")
print(f"  S5 logistique  : sigma_iter = {sigma_iter_S5:.5f}  (> controle)")
print()
print("Conclusion honnete : S4 ET S5 dissipent tous les deux (au-dessus du")
print("controle uniforme). I_thermo seul NE discrimine PAS l'agent du dissipateur.")
print("C'est repair_gain (section 4) qui discrimine.")

Bilan de flux (proxy dissipation NESS) :
  S4 Gray-Scott  : sigma_flux = 0.00530  (> controle 0.00000)
  S5 logistique  : sigma_iter = 0.22082  (> controle)

Conclusion honnete : S4 ET S5 dissipent tous les deux (au-dessus du
controle uniforme). I_thermo seul NE discrimine PAS l'agent du dissipateur.
C'est repair_gain (section 4) qui discrimine.


In [8]:
# Exercice 1 a completer
# Conseil : bouclez sur warmup_values, re-seed + run + measure repair_gain(S4).
# Indice : un motif non forme a peu de structure a reparer -> repair_gain faible.
print("Exercice a completer")

Exercice a completer


### Exercice 2 — Un autre dissipateur (Brusselator limite-cycle)
Construisez un substrat S5c = Brusselator en limite-cycle (oscillateur chimique
auto-entretenu). Mesurez son `repair_gain`. Hypothèse : contrairement à S5
(logistique forcé, pas de consigne), le Brusselator auto-entretenu devrait avoir
un `repair_gain > 0` — confirmez ou infirmez.

In [9]:
# Exercice 2 a completer
# Conseil : Brusselator dx/dt = A - (B+1)x + x^2 y, dy/dt = Bx - x^2 y, B > 1+sqrt(A).
# Mesurez repair_gain apres ablation d'une region oscillante.
print("Exercice a completer")

Exercice a completer


### Exercice 3 — Graduation ENJEU-2
Mesurez `repair_gain` pour S1 (tri), S2 (bistable), S3 (Axelrod) en plus de S4/S5.
Classez les 5 substrats par `repair_gain` décroissant. La graduation est-elle
cohérente avec l'intuition (auto-maintien fort → faible → dissipateur pur) ?

In [10]:
# Exercice 3 a completer
# Conseil : pour chaque substrat, ablate + measure repair_gain vs controle passif.
# Comparez aux Graduations ENJEU-2 documentees.
print("Exercice a completer")

Exercice a completer


## 7. Ponts avec la série

| Direction | Lien | Relation |
|-----------|------|----------|
| ↔ ICT-9 | [ICT-9 — Agence morphologique](ICT-9-AgencyRegeneration.ipynb) | `agency` module (`ablate`, `recovery_score`, `repair_gain`) |
| ↔ ICT-18 | [ICT-18 — Flèche du temps](ICT-18-ArrowOfTimeReversibilization.ipynb) | `I_thermo` (Schnakenberg) — faux-positif S5 oscillateur forcé |
| ↗ ICT-20+ | *(à venir)* | Graduation ENJEU-2 multi-substrats |

---

**Ce qu'il faut retenir.** Dissiper n'est pas réparer. La batterie de l'ENJEU
mesure le **gain de réparation** — le contrast do-calculus entre la dynamique
complète et un contrôle privé du mécanisme réparateur — comme l'instrument
falsifiable qui sépare l'agent du pur dissipateur. Un système qui répare activement
sa forme après une agression manifeste de l'agence ; un système qui dissipe sans
réparer n'en manifeste pas, aussi intense que soit son flux.